** Notes: **
* This ipynb parses the election data from ./VoteCountByStates/
* into a panel (long-form) of state-district-county-politician level data
* stored in **./output/counties-election-results.csv**

* Also stores a list of the political candidates with observed individual characteristics (e.g. gender and race) into **./output/politicians.csv**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os.path
import os

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 100)

In [ ]:
folderpath = os.path.join('VoteCountByStatesData')
outputTo = os.path.join('output')

In [ ]:
states = list()
for file in os.listdir(folderpath):
    _fileName, _fileExt = os.path.splitext(file)
    
    if '~$' in _fileName:
            continue
    
    if 'xlsx' in _fileExt:
        states.append(_fileName)

In [ ]:
def third_party(party):
    if (party != 'dem') and (party != 'rep'):
        return 'yes'
    else:
        return 'no'

### Parsing county and individual politician data

In [ ]:
dfCounties = pd.DataFrame(columns=['state', 'state full', 'district', 'county', 'politician', 'votes', 
                                   'open seat', 'no main challenger', 'multiple rep', 'multiple dem', 'third party', 
                                   'all women', 'all men', 'same gender', 'woman challenger'])
dfIndividual = pd.DataFrame(columns=['politician', 'party', 'third party', 'incumbent', 'gender', 
                                     'black', 'state', 'state full', 'district'])

In [ ]:
for state in states:
    
    fileName = os.path.join(folderpath, state + '.xlsx')
    excelFile = pd.ExcelFile(fileName)
    excelSheets = excelFile.sheet_names
    
    for sheet in excelSheets:
        _dfState = pd.read_excel(fileName, sheet)
        
        _dfState, _dfInd = _dfState.iloc[:-4], _dfState.iloc[-4:]
                
        # store individual politician data
        _dfInd.columns = [''] + _dfInd.columns.tolist()[1:]
        _dfInd = _dfInd.transpose()
        _dfInd.columns = _dfInd.iloc[0]
        _dfInd = _dfInd.reindex(_dfInd.index.drop(''))
        
        _dfInd['third party'] = _dfInd['party'].apply(third_party)
        _dfInd['state'] = state.split('-')[0]
        _dfInd['state full'] = state.split('-')[1]
        _dfInd['district'] = sheet
        _dfInd.index.name = 'politician'
        _dfInd.reset_index(inplace=True)
        dfIndividual = dfIndividual.append(_dfInd, ignore_index=True, sort=False)
        
        # check if there are multiple main challengers, no main challengers, openseat, third party challengers
        _countRep = _dfInd['party'].tolist().count('rep')
        _countDem = _dfInd['party'].tolist().count('dem')
            
        if _countRep > 1:
            multiRep = True
        else:
            multiRep = False
            
        if _countDem > 1:
            multiDem = True 
        else:
            multiDem = False
            
        if _countRep + _countDem < 2:
            noMainChallenge = True
        else:
            noMainChallenge = False
            
        if _dfInd['incumbent'].tolist().count('yes')==0:
            openSeat = True
        else:
            openSeat = False
            
        if _dfInd['third party'].tolist().count('yes') > 0:
            thirdparty = True
        else:
            thirdparty = False
                                    
        assert _dfInd['incumbent'].tolist().count('yes') < 2
        try:
            assert _countRep + _countDem > 0
        except AssertionError:
            print('_countRep + _countDem: ', state, sheet)
        
        # parse state data
        _dfState = pd.melt(_dfState, id_vars=['county'], 
                           value_vars=_dfState.columns.tolist()[1:], 
                           var_name='politician', value_name='votes')
        
        _dfState['state'] = state.split('-')[0]
        _dfState['state full'] = state.split('-')[1]        
        _dfState['district'] = sheet
        if openSeat:
            _dfState['open seat'] = 'yes'
        else:
            _dfState['open seat'] = 'no'
            
        if noMainChallenge:
            _dfState['no main challenger'] = 'yes'
        else: 
            _dfState['no main challenger'] = 'no'
            
        if multiRep:
            _dfState['multiple rep'] = 'yes'
        else:
            _dfState['multiple rep'] = 'no'
            
        if multiDem:
            _dfState['multiple dem'] = 'yes'
        else:
            _dfState['multiple dem'] = 'no'
            
        if thirdparty:
            _dfState['third party'] = 'yes'
        else:
            _dfState['third party'] = 'no'
            
        _countWoman = _dfInd['gender'].tolist().count('female')            
        _countPoliticians = len(_dfInd['gender'].tolist())
        if _countWoman == _countPoliticians:
            _dfState['all women'] = 'yes'
            _dfState['same gender'] = 'yes'
        else:
            _dfState['all women'] = 'no'
            _dfState['same gender'] = 'no'
            
        if _countWoman == 0:
            _dfState['all men'] = 'yes'
            _dfState['same gender'] = 'yes'
        else:
            _dfState['all men'] = 'no'
            _dfState['same gender'] = 'no'
            
        if (_countWoman > 0) & (_countWoman < _countPoliticians):
            _dfState['woman challenger'] = 'yes'
        else:
            _dfState['woman challenger'] = 'no'
                    
        dfCounties = dfCounties.append(_dfState, ignore_index=True, sort=False)
        dfCounties.sort_values(['state', 'district','county', 'politician'], inplace=True)
        
# strip county names on both sides of whitespaces
dfCounties['county'] = dfCounties['county'].str.strip()
        
dfCounties.to_csv(os.path.join(outputTo, 'counties-election-results.csv'), index=False)

### Cleaning up individual politician data

In [ ]:
assert len(set(dfIndividual['gender'].tolist())) == 2
assert len(set(dfIndividual['black'].tolist())) == 2
assert len(set(dfIndividual['incumbent'].tolist())) == 2

dfIndividual['write-in'] = dfIndividual['party'].apply(lambda x: 'yes' if x == 'wri' else 'no')

mask = dfIndividual['incumbent'].isnull()
dfIndividual.loc[mask, 'incumbent'] = 'no'

mask = dfIndividual['gender'].isnull()
dfIndividual.loc[mask, 'gender'] = 'male'

mask = dfIndividual['black'].isnull()
dfIndividual.loc[mask, 'black'] = 'no'

dfIndividual.to_csv(os.path.join(outputTo, 'politicians.csv'), index=False)

### List of distinct counties

In [ ]:
listCounties = dfCounties[['state', 'state full', 'county']]

listCounties.drop_duplicates(listCounties.columns, keep='first', inplace=True)

listCounties['county'] = listCounties.iloc[:, 2].str.lower()

listCounties.to_csv(os.path.join(outputTo, 'county-list.csv'), index=False)